# XLKD-06: Dense Mapped Logit-KD three-seed aggregate

Tổng hợp xác nhận độc lập cho T1 ExpansionNet v2 → S1
MobileNetV3-Large tại ba seed đã đăng ký trước: `42, 123, 2026`.

Nguyên tắc khóa:

- chỉ dùng Validation để tổng hợp và kết luận;
- gate giữ nguyên: mỗi seed phải có `ΔCIDEr-D ≥ 0.001`;
- protocol/loss/production-contract phải đồng nhất tuyệt đối;
- kết quả âm vẫn được đóng gói, không nới gate hậu nghiệm;
- Held-out Test luôn `SEALED / NOT USED`;
- không huấn luyện và không đọc ảnh trong notebook này.

## Kaggle inputs

1. Output XLKD-03 chứa `final_kd_roadmap.json`.
2. Output XLKD-04 seed 42 đã hoàn tất.
3. Output XLKD-05 seed 123 đã hoàn tất.
4. Output XLKD-05 seed 2026 đã hoàn tất.

Runtime CPU, Internet Off.


## Cell 1 — Three-run authorization và manifest integrity


In [ ]:
import csv
import hashlib
import json
import math
import os
import platform
import statistics
import time
from datetime import datetime, timezone
from pathlib import Path

STARTED = time.perf_counter()
SEEDS = (42, 123, 2026)
EXPERIMENT_ID = "XLKD_EXPANSIONNET_S1_DENSE_MAPPED_LOGITKD"
INPUT_ROOT = Path(os.environ.get("KLCN_INPUT_ROOT", "/kaggle/input"))
NOTEBOOK_ROOT = (
    INPUT_ROOT / "notebooks"
    if (INPUT_ROOT / "notebooks").is_dir()
    else INPUT_ROOT
)
OUT = Path(os.environ.get(
    "KLCN_OUTPUT_ROOT",
    "/kaggle/working//"
    "xlkd06_dense_mapped_logit_kd_three_seed_aggregate",
))
AUDIT = OUT / "audits"
RESULT = OUT / "results"
FIGURE = OUT / "figures"
MANIFEST = OUT / "manifests"
for directory in [OUT, AUDIT, RESULT, FIGURE, MANIFEST]:
    directory.mkdir(parents=True, exist_ok=True)

def now():
    return datetime.now(timezone.utc).isoformat()

def read_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)

def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(
            lambda: handle.read(8 * 1024 * 1024), b""
        ):
            digest.update(block)
    return digest.hexdigest()

def resolve_inside(root, relative_path):
    root = Path(root).resolve()
    target = (root / relative_path).resolve()
    assert target.is_relative_to(root), (
        root, relative_path, target
    )
    return target

discovery = {}
for candidate in NOTEBOOK_ROOT.rglob("*"):
    if candidate.is_file():
        discovery.setdefault(candidate.name, []).append(
            candidate.resolve()
        )

def exactly_one(name):
    matches = sorted(discovery.get(name, []))
    assert len(matches) == 1, (
        f"Cần đúng một {name} dưới {NOTEBOOK_ROOT}; "
        f"tìm thấy {len(matches)}: {matches}"
    )
    return matches[0]

def verify_manifest(root, reference):
    path = resolve_inside(root, reference["relative_path"])
    assert path.is_file()
    assert sha256(path) == reference["sha256"]
    with path.open(
        "r", encoding="utf-8", newline=""
    ) as handle:
        rows = list(csv.DictReader(handle))
    if "rows" in reference:
        assert len(rows) == int(reference["rows"])
    for row in rows:
        artifact = resolve_inside(root, row["relative_path"])
        assert artifact.is_file()
        assert artifact.stat().st_size == int(row["size_bytes"])
        assert sha256(artifact) == row["sha256"]
    return rows

marker_names = {
    42: (
        "XLKD04_DENSE_MAPPED_LOGIT_KD_"
        "SEED42_FULL_RUN_COMPLETE.json"
    ),
    123: (
        "XLKD05_DENSE_MAPPED_LOGIT_KD_"
        "SEED123_FULL_RUN_COMPLETE.json"
    ),
    2026: (
        "XLKD05_DENSE_MAPPED_LOGIT_KD_"
        "SEED2026_FULL_RUN_COMPLETE.json"
    ),
}
RUNS = {}
total_source_rows = 0
for seed in SEEDS:
    marker_path = exactly_one(marker_names[seed])
    marker = read_json(marker_path)
    assert marker["status"] == "complete"
    assert marker["experiment_id"] == EXPERIMENT_ID
    assert marker["method"] == (
        "cross-vocabulary dense mapped Logit-KD"
    )
    assert int(marker["seed"]) == seed
    assert float(marker["temperature"]) == 4.0
    assert float(marker["alpha_human_ce"]) == 0.5
    assert float(marker["alpha_kd"]) == 0.5
    assert int(marker["completed_epochs"]) >= 1
    assert marker["best_result"] is not None
    assert marker["comparison"] is not None
    assert marker["manifest_verification"] == "all_passed"
    assert marker["selection_split"] == "validation"
    assert marker["test_set_status"] == "sealed"
    assert marker["test_set_used"] is False
    if seed == 42:
        assert marker["confirmation_seeds_authorized"] is True
    else:
        assert marker["confirmation_run"] is True
        assert marker["ready_for_xlkd06_aggregate"] is True
    rows = verify_manifest(
        marker_path.parent, marker["manifest"]["csv"]
    )
    assert len(rows) == int(marker["manifest_rows"])
    total_source_rows += len(rows)
    RUNS[seed] = {
        "root": marker_path.parent,
        "marker_path": marker_path,
        "marker": marker,
        "manifest_rows": rows,
    }

roadmap_path = exactly_one("final_kd_roadmap.json")
roadmap = read_json(roadmap_path)
assert roadmap["status"] == "locked"
assert roadmap["governance"]["gate_before_next_phase"] is True
assert roadmap["governance"][
    "no_posthoc_threshold_relaxation"
] is True
assert roadmap["governance"][
    "validation_only_for_selection"
] is True
assert roadmap["governance"][
    "test_open_once_after_final_method_lock"
] is True
assert "XLKD-06 response-KD three-seed aggregate" in (
    roadmap["stages"]
)

contract_hashes = {
    RUNS[seed]["marker"]["production_contract"]["source_sha256"]
    for seed in SEEDS
}
assert len(contract_hashes) == 1

write_json(AUDIT / "xlkd06_cell01_inputs.json", {
    "schema_version": "xlkd06_cell01_v1",
    "created_at_utc": now(),
    "status": "passed",
    "seeds": list(SEEDS),
    "source_runs": "3 / 3",
    "source_manifest_rows": total_source_rows,
    "source_manifests_verified": True,
    "production_contract_sha256": next(iter(contract_hashes)),
    "roadmap_status": "locked",
    "no_posthoc_threshold_relaxation": True,
    "training_started": False,
    "image_payload_read": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "python": platform.python_version(),
    "elapsed_seconds": time.perf_counter() - STARTED,
})
print("=" * 78)
print("XLKD-06 CELL 1 — INPUT AUTHORIZATION")
print("=" * 78)
print("Status                    : PASSED")
print("Source runs               : 3 / 3 VERIFIED")
print("Source manifest rows      :", total_source_rows)
print("Production contract       : EXACT")
print("Roadmap / gate policy     : LOCKED")
print("Training / images         : NOT STARTED / NOT READ")
print("Test Set status           : SEALED")
print("Progress                  : 1 / 5 code cells")


## Cell 2 — Paired seed metrics và descriptive aggregate


In [ ]:
CELL_STARTED = time.perf_counter()
METRICS = (
    "BLEU-1", "BLEU-4", "METEOR", "ROUGE-L", "CIDEr-D"
)
SEED_ROWS = []
CONTRACTS = {}

for seed in SEEDS:
    marker = RUNS[seed]["marker"]
    comparison = marker["comparison"]
    baseline = comparison[f"baseline_seed{seed}"]
    kd = comparison[f"mapped_logit_kd_seed{seed}"]
    delta = comparison["delta_kd_minus_baseline"]
    assert comparison["selection_primary"] == "CIDEr-D"
    assert comparison["selection_tiebreakers"] == [
        "BLEU-4", "lower CE", "earlier epoch"
    ]
    assert comparison["selection_split"] == "validation"
    assert comparison["test_set_status"] == "sealed"
    assert comparison["test_set_used"] is False
    assert int(kd["best_epoch"]) == int(
        marker["best_result"]["epoch"]
    )

    best_checkpoint = resolve_inside(
        RUNS[seed]["root"],
        marker["best_checkpoint"]["relative_path"],
    )
    assert sha256(best_checkpoint) == (
        marker["best_checkpoint"]["sha256"]
    )

    row = {
        "seed": seed,
        "completed_epochs": int(marker["completed_epochs"]),
        "best_epoch": int(kd["best_epoch"]),
    }
    for metric in METRICS:
        b = float(baseline[metric])
        k = float(kd[metric])
        d = float(delta[metric])
        assert math.isclose(k - b, d, abs_tol=1e-12)
        row[f"baseline_{metric}"] = b
        row[f"kd_{metric}"] = k
        row[f"delta_{metric}"] = d
    baseline_ce = float(baseline["CE"])
    kd_ce = float(kd["CE"])
    delta_ce = float(delta["CE"])
    assert math.isclose(
        kd_ce - baseline_ce, delta_ce, abs_tol=1e-12
    )
    row["baseline_CE"] = baseline_ce
    row["kd_CE"] = kd_ce
    row["delta_CE"] = delta_ce
    row["cider_gate_plus_0_001"] = (
        row["delta_CIDEr-D"] >= 0.001
    )
    assert bool(
        comparison["kd_improves_cider_by_0_001"]
    ) == row["cider_gate_plus_0_001"]
    SEED_ROWS.append(row)
    CONTRACTS[seed] = {
        "experiment_id": marker["experiment_id"],
        "method": marker["method"],
        "temperature": float(marker["temperature"]),
        "alpha_human_ce": float(marker["alpha_human_ce"]),
        "alpha_kd": float(marker["alpha_kd"]),
        "production_contract_sha256": (
            marker["production_contract"]["source_sha256"]
        ),
        "selection_primary": comparison["selection_primary"],
        "selection_tiebreakers": (
            comparison["selection_tiebreakers"]
        ),
        "selection_split": comparison["selection_split"],
        "test_set_status": marker["test_set_status"],
        "test_set_used": marker["test_set_used"],
    }

PER_SEED_CSV = RESULT / "xlkd06_three_seed_paired_metrics.csv"
with PER_SEED_CSV.open(
    "w", encoding="utf-8", newline=""
) as handle:
    writer = csv.DictWriter(
        handle, fieldnames=list(SEED_ROWS[0])
    )
    writer.writeheader()
    writer.writerows(SEED_ROWS)

def describe(values):
    values = [float(value) for value in values]
    mean = statistics.mean(values)
    sample_std = statistics.stdev(values)
    return {
        "mean": mean,
        "sample_std": sample_std,
        "population_std": statistics.pstdev(values),
        "coefficient_of_variation_percent": (
            100.0 * sample_std / abs(mean)
            if mean != 0.0 else None
        ),
        "minimum": min(values),
        "maximum": max(values),
    }

AGGREGATE = {}
for metric in (*METRICS, "CE"):
    AGGREGATE[metric] = {
        "baseline": describe(
            row[f"baseline_{metric}"] for row in SEED_ROWS
        ),
        "kd": describe(
            row[f"kd_{metric}"] for row in SEED_ROWS
        ),
        "paired_delta": describe(
            row[f"delta_{metric}"] for row in SEED_ROWS
        ),
    }
relative_cider_gain = [
    100.0 * row["delta_CIDEr-D"] / row["baseline_CIDEr-D"]
    for row in SEED_ROWS
]
AGGREGATE["relative_CIDEr-D_gain_percent"] = describe(
    relative_cider_gain
)

cider_wins = sum(
    row["cider_gate_plus_0_001"] for row in SEED_ROWS
)
SUMMARY_PATH = RESULT / "xlkd06_three_seed_summary.json"
write_json(SUMMARY_PATH, {
    "schema_version": "xlkd06_three_seed_summary_v1",
    "created_at_utc": now(),
    "experiment_id": EXPERIMENT_ID,
    "seeds": list(SEEDS),
    "per_seed": SEED_ROWS,
    "aggregate": AGGREGATE,
    "paired_consistency": {
        "cider_delta_at_least_0_001": f"{cider_wins} / 3",
        "one_sided_exact_sign_probability_if_3_of_3": (
            0.125 if cider_wins == 3 else None
        ),
        "interpretation": (
            "Descriptive only; n=3 is insufficient for a "
            "strong inferential claim."
        ),
    },
    "selection_split": "validation",
    "test_set_status": "sealed",
    "test_set_used": False,
})
write_json(AUDIT / "xlkd06_cell02_metrics.json", {
    "schema_version": "xlkd06_cell02_v1",
    "created_at_utc": now(),
    "status": "passed",
    "finite_metrics": all(
        math.isfinite(value)
        for row in SEED_ROWS
        for value in row.values()
        if isinstance(value, float)
    ),
    "paired_seed_alignment": "3 / 3",
    "cider_gate_count": cider_wins,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - CELL_STARTED,
})
print("=" * 78)
print("XLKD-06 CELL 2 — PAIRED RESULTS")
print("=" * 78)
for row in SEED_ROWS:
    print(
        f"seed={row['seed']:4d} | "
        f"CIDEr-D {row['baseline_CIDEr-D']:.6f} -> "
        f"{row['kd_CIDEr-D']:.6f} "
        f"({row['delta_CIDEr-D']:+.6f}) | "
        f"BLEU-4 {row['delta_BLEU-4']:+.6f} | "
        f"CE {row['delta_CE']:+.6f}"
    )
print("CIDEr-D gate             :", f"{cider_wins} / 3")
print(
    "Mean paired CIDEr delta  :",
    f"{AGGREGATE['CIDEr-D']['paired_delta']['mean']:+.6f}",
)
print("Test Set status           : SEALED")
print("Progress                  : 2 / 5 code cells")


## Cell 3 — Aggregate learning curves và figures


In [ ]:
CELL_STARTED = time.perf_counter()
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HISTORY_BY_SEED = {}
for seed in SEEDS:
    phase = "xlkd04" if seed == 42 else "xlkd05"
    history_path = (
        RUNS[seed]["root"] / "logs"
        / f"{phase}_dense_mapped_seed{seed}_epoch_history.csv"
    )
    assert history_path.is_file()
    with history_path.open(
        "r", encoding="utf-8", newline=""
    ) as handle:
        rows = list(csv.DictReader(handle))
    assert len(rows) == int(
        RUNS[seed]["marker"]["completed_epochs"]
    )
    assert [int(row["epoch"]) for row in rows] == list(
        range(1, len(rows) + 1)
    )
    HISTORY_BY_SEED[seed] = rows

common_epochs = min(
    len(rows) for rows in HISTORY_BY_SEED.values()
)
curve_metrics = (
    "train_combined_loss",
    "train_human_ce",
    "train_mapped_kl_t2",
    "validation_ce",
    "BLEU-4",
    "CIDEr-D",
)
CURVE_ROWS = []
for epoch in range(1, common_epochs + 1):
    row = {"epoch": epoch}
    for metric in curve_metrics:
        values = [
            float(HISTORY_BY_SEED[seed][epoch - 1][metric])
            for seed in SEEDS
        ]
        row[f"{metric}_mean"] = statistics.mean(values)
        row[f"{metric}_sample_std"] = statistics.stdev(values)
    CURVE_ROWS.append(row)

CURVE_CSV = RESULT / "xlkd06_three_seed_epoch_curves.csv"
with CURVE_CSV.open(
    "w", encoding="utf-8", newline=""
) as handle:
    writer = csv.DictWriter(
        handle, fieldnames=list(CURVE_ROWS[0])
    )
    writer.writeheader()
    writer.writerows(CURVE_ROWS)

colors = {42: "#2563eb", 123: "#16a34a", 2026: "#dc2626"}
epochs = list(range(1, common_epochs + 1))

fig, ax = plt.subplots(figsize=(10, 6))
for seed in SEEDS:
    ax.plot(
        epochs,
        [
            float(row["CIDEr-D"])
            for row in HISTORY_BY_SEED[seed][:common_epochs]
        ],
        label=f"Seed {seed}",
        color=colors[seed],
        linewidth=1.8,
    )
ax.set_title(
    "T1 → S1 Dense Mapped Logit-KD: Validation CIDEr-D"
)
ax.set_xlabel("Epoch")
ax.set_ylabel("CIDEr-D")
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
CIDER_FIGURE = FIGURE / "xlkd06_cider_by_epoch.png"
fig.savefig(CIDER_FIGURE, dpi=180)
plt.close(fig)

fig, ax = plt.subplots(figsize=(10, 6))
for seed in SEEDS:
    ax.plot(
        epochs,
        [
            float(row["validation_ce"])
            for row in HISTORY_BY_SEED[seed][:common_epochs]
        ],
        label=f"Seed {seed}",
        color=colors[seed],
        linewidth=1.8,
    )
ax.set_title(
    "T1 → S1 Dense Mapped Logit-KD: Validation CE"
)
ax.set_xlabel("Epoch")
ax.set_ylabel("Cross-Entropy (lower is better)")
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
CE_FIGURE = FIGURE / "xlkd06_validation_ce_by_epoch.png"
fig.savefig(CE_FIGURE, dpi=180)
plt.close(fig)

fig, ax = plt.subplots(figsize=(9, 6))
positions = list(range(len(SEEDS)))
width = 0.34
ax.bar(
    [value - width / 2 for value in positions],
    [row["baseline_CIDEr-D"] for row in SEED_ROWS],
    width,
    label="CE-only baseline",
    color="#94a3b8",
)
ax.bar(
    [value + width / 2 for value in positions],
    [row["kd_CIDEr-D"] for row in SEED_ROWS],
    width,
    label="Mapped Logit-KD",
    color="#2563eb",
)
ax.set_xticks(positions, [str(seed) for seed in SEEDS])
ax.set_xlabel("Seed")
ax.set_ylabel("Validation CIDEr-D")
ax.set_title("Paired baseline vs. mapped Logit-KD")
ax.grid(axis="y", alpha=0.25)
ax.legend()
fig.tight_layout()
PAIRED_FIGURE = FIGURE / "xlkd06_baseline_vs_kd_cider.png"
fig.savefig(PAIRED_FIGURE, dpi=180)
plt.close(fig)

for figure in [CIDER_FIGURE, CE_FIGURE, PAIRED_FIGURE]:
    assert figure.is_file()
    assert figure.stat().st_size > 1000

write_json(AUDIT / "xlkd06_cell03_curves.json", {
    "schema_version": "xlkd06_cell03_v1",
    "created_at_utc": now(),
    "status": "passed",
    "seed_histories": "3 / 3",
    "common_epochs": common_epochs,
    "curve_rows": len(CURVE_ROWS),
    "figures": {
        path.name: {
            "size_bytes": path.stat().st_size,
            "sha256": sha256(path),
        }
        for path in [
            CIDER_FIGURE, CE_FIGURE, PAIRED_FIGURE
        ]
    },
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - CELL_STARTED,
})
print("=" * 78)
print("XLKD-06 CELL 3 — CURVES AND FIGURES")
print("=" * 78)
print("Status                    : PASSED")
print("Seed histories            : 3 / 3")
print("Common epochs             :", common_epochs)
print("Aggregate curve rows      :", len(CURVE_ROWS))
print("Figures                   : 3 / 3 CREATED")
print("Test Set status           : SEALED")
print("Progress                  : 3 / 5 code cells")


## Cell 4 — Prespecified reproduction gate và next authorization


In [ ]:
CELL_STARTED = time.perf_counter()
cider_gate_count = sum(
    row["cider_gate_plus_0_001"] for row in SEED_ROWS
)
protocol_exact = all(
    CONTRACTS[seed] == CONTRACTS[42]
    for seed in (123, 2026)
)
finite_metrics = all(
    math.isfinite(value)
    for row in SEED_ROWS
    for value in row.values()
    if isinstance(value, float)
)
REPRODUCTION_PASSED = (
    cider_gate_count == 3
    and protocol_exact
    and finite_metrics
)
REPRODUCTION_STATUS = (
    "passed"
    if REPRODUCTION_PASSED
    else "negative_result_retained"
)

CONCLUSION_PATH = RESULT / "xlkd06_reproduction_conclusion.json"
write_json(CONCLUSION_PATH, {
    "schema_version": "xlkd06_reproduction_conclusion_v1",
    "created_at_utc": now(),
    "status": REPRODUCTION_STATUS,
    "experiment_id": EXPERIMENT_ID,
    "teacher": "T1 — ExpansionNet v2",
    "student": "S1 — MobileNetV3-Large",
    "method": "cross-vocabulary dense mapped response/logit KD",
    "seeds": list(SEEDS),
    "prespecified_gate": {
        "metric": "Validation CIDEr-D",
        "minimum_paired_delta_per_seed": 0.001,
        "required_seed_count": "3 / 3",
        "observed_seed_count": f"{cider_gate_count} / 3",
        "protocol_consistency_required": "exact",
        "protocol_consistency_observed": (
            "exact" if protocol_exact else "mismatch"
        ),
        "posthoc_relaxation": False,
    },
    "mean_results": {
        "baseline_cider_d": (
            AGGREGATE["CIDEr-D"]["baseline"]["mean"]
        ),
        "kd_cider_d": AGGREGATE["CIDEr-D"]["kd"]["mean"],
        "paired_cider_delta": (
            AGGREGATE["CIDEr-D"]["paired_delta"]["mean"]
        ),
        "relative_cider_gain_percent": (
            AGGREGATE[
                "relative_CIDEr-D_gain_percent"
            ]["mean"]
        ),
        "baseline_bleu_4": (
            AGGREGATE["BLEU-4"]["baseline"]["mean"]
        ),
        "kd_bleu_4": AGGREGATE["BLEU-4"]["kd"]["mean"],
        "paired_bleu_4_delta": (
            AGGREGATE["BLEU-4"]["paired_delta"]["mean"]
        ),
        "baseline_validation_ce": (
            AGGREGATE["CE"]["baseline"]["mean"]
        ),
        "kd_validation_ce": AGGREGATE["CE"]["kd"]["mean"],
        "paired_validation_ce_delta": (
            AGGREGATE["CE"]["paired_delta"]["mean"]
        ),
    },
    "fairness_disclosure": (
        "Post-baseline KD fine-tuning adds up to 20 epochs after "
        "the 20-epoch CE baseline; total budgets are not equal."
    ),
    "statistical_caution": (
        "Only three prespecified seeds are available. Report "
        "mean, sample SD and paired deltas descriptively."
    ),
    "selection_split": "validation",
    "test_set_status": "sealed",
    "test_set_used": False,
})

if REPRODUCTION_PASSED:
    NEXT_AUTHORIZATION = OUT / (
        "MLKD00_FEATURE_KD_PREFLIGHT_AUTHORIZED.json"
    )
    write_json(NEXT_AUTHORIZATION, {
        "schema_version": "mlkd00_feature_kd_preflight_authorized_v1",
        "created_at_utc": now(),
        "status": "authorized",
        "source_experiment": EXPERIMENT_ID,
        "source_reproduction_status": "passed",
        "required_next": (
            "MLKD-00/01 feature tap, adapter and fixed-batch "
            "preflight"
        ),
        "method_changes_allowed": False,
        "held_out_test_may_be_opened": False,
        "test_set_status": "sealed",
        "test_set_used": False,
    })
else:
    NEXT_AUTHORIZATION = None
    NEGATIVE_MARKER = OUT / (
        "XLKD06_RESPONSE_KD_NEGATIVE_RESULT_RETAINED.json"
    )
    write_json(NEGATIVE_MARKER, {
        "schema_version": (
            "xlkd06_response_kd_negative_result_retained_v1"
        ),
        "created_at_utc": now(),
        "status": "negative_result_retained",
        "cider_gate_count": f"{cider_gate_count} / 3",
        "next_phase_authorized": False,
        "posthoc_threshold_relaxation": False,
        "test_set_status": "sealed",
        "test_set_used": False,
    })

write_json(AUDIT / "xlkd06_cell04_gate.json", {
    "schema_version": "xlkd06_cell04_v1",
    "created_at_utc": now(),
    "status": "passed",
    "reproduction_status": REPRODUCTION_STATUS,
    "cider_gate_count": cider_gate_count,
    "protocol_exact": protocol_exact,
    "finite_metrics": finite_metrics,
    "next_phase_authorized": REPRODUCTION_PASSED,
    "negative_result_retained": not REPRODUCTION_PASSED,
    "posthoc_threshold_relaxation": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - CELL_STARTED,
})
print("=" * 78)
print("XLKD-06 CELL 4 — PRESPECIFIED GATE")
print("=" * 78)
print("Status                    :", REPRODUCTION_STATUS.upper())
print("CIDEr-D delta >= 0.001    :", f"{cider_gate_count} / 3")
print("Protocol consistency      :", (
    "EXACT" if protocol_exact else "MISMATCH"
))
print("Finite metrics            :", finite_metrics)
print("Posthoc relaxation        : False")
print("Negative result retained  :", not REPRODUCTION_PASSED)
print("MLKD-00/01                :", (
    "AUTHORIZED" if REPRODUCTION_PASSED else "NOT AUTHORIZED"
))
print("Test Set status           : SEALED")
print("Progress                  : 4 / 5 code cells")


## Cell 5 — Formal package và manifest


In [ ]:
CELL_STARTED = time.perf_counter()
MANIFEST_CSV = MANIFEST / "xlkd06_manifest.csv"
MANIFEST_JSON = MANIFEST / "xlkd06_manifest.json"
SHA256SUMS = MANIFEST / "XLKD06_SHA256SUMS.txt"
FINAL_MARKER = OUT / (
    "XLKD06_DENSE_MAPPED_LOGIT_KD_"
    "THREE_SEED_AGGREGATE_COMPLETE.json"
)

artifact_paths = sorted(
    [
        path for path in OUT.rglob("*")
        if path.is_file()
        and path not in {
            MANIFEST_CSV,
            MANIFEST_JSON,
            SHA256SUMS,
            FINAL_MARKER,
        }
        and not path.name.endswith(".tmp")
    ],
    key=lambda path: path.relative_to(OUT).as_posix(),
)
manifest_rows = [
    {
        "relative_path": path.relative_to(OUT).as_posix(),
        "size_bytes": path.stat().st_size,
        "sha256": sha256(path),
    }
    for path in artifact_paths
]
with MANIFEST_CSV.open(
    "w", encoding="utf-8", newline=""
) as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=[
            "relative_path", "size_bytes", "sha256"
        ],
    )
    writer.writeheader()
    writer.writerows(manifest_rows)
write_json(MANIFEST_JSON, {
    "schema_version": "xlkd06_manifest_v1",
    "created_at_utc": now(),
    "artifact_count": len(manifest_rows),
    "artifacts": manifest_rows,
})
SHA256SUMS.write_text(
    "".join(
        f'{row["sha256"]}  {row["relative_path"]}\n'
        for row in manifest_rows
    ),
    encoding="utf-8",
)
for row in manifest_rows:
    artifact = resolve_inside(OUT, row["relative_path"])
    assert artifact.stat().st_size == int(row["size_bytes"])
    assert sha256(artifact) == row["sha256"]

write_json(FINAL_MARKER, {
    "schema_version": (
        "xlkd06_three_seed_aggregate_complete_v1"
    ),
    "created_at_utc": now(),
    "status": "complete",
    "experiment_id": EXPERIMENT_ID,
    "seeds": list(SEEDS),
    "source_runs_verified": "3 / 3",
    "source_manifest_rows_verified": total_source_rows,
    "protocol_consistency": (
        "exact" if protocol_exact else "mismatch"
    ),
    "reproduction_status": REPRODUCTION_STATUS,
    "cider_gate": f"{cider_gate_count} / 3",
    "posthoc_threshold_relaxation": False,
    "summary": {
        "relative_path": SUMMARY_PATH.relative_to(OUT).as_posix(),
        "sha256": sha256(SUMMARY_PATH),
    },
    "conclusion": {
        "relative_path": (
            CONCLUSION_PATH.relative_to(OUT).as_posix()
        ),
        "sha256": sha256(CONCLUSION_PATH),
    },
    "next_authorization": (
        {
            "relative_path": (
                NEXT_AUTHORIZATION.relative_to(OUT).as_posix()
            ),
            "sha256": sha256(NEXT_AUTHORIZATION),
        }
        if NEXT_AUTHORIZATION is not None else None
    ),
    "manifest": {
        "csv": {
            "relative_path": (
                MANIFEST_CSV.relative_to(OUT).as_posix()
            ),
            "sha256": sha256(MANIFEST_CSV),
            "rows": len(manifest_rows),
        },
        "json": {
            "relative_path": (
                MANIFEST_JSON.relative_to(OUT).as_posix()
            ),
            "sha256": sha256(MANIFEST_JSON),
        },
        "sha256sums": {
            "relative_path": (
                SHA256SUMS.relative_to(OUT).as_posix()
            ),
            "sha256": sha256(SHA256SUMS),
        },
    },
    "manifest_rows": len(manifest_rows),
    "manifest_verification": "all_passed",
    "selection_split": "validation",
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - CELL_STARTED,
})
print("=" * 78)
print("XLKD-06 CELL 5 — FORMAL PACKAGE")
print("=" * 78)
print("Status                    : COMPLETE")
print("Source runs               : 3 / 3 VERIFIED")
print("Reproduction status       :", REPRODUCTION_STATUS.upper())
print("Manifest rows             :", len(manifest_rows))
print("Manifest verification     : ALL PASSED")
print("Figures                   : 3 / 3")
print("Selection split           : Validation")
print("Test Set status           : SEALED")
print("Next                      :", (
    "MLKD-00/01"
    if REPRODUCTION_PASSED
    else "Review retained negative result"
))
print("Progress                  : 5 / 5 code cells")
print("Marker                    :", FINAL_MARKER)
